# 疾患の治療標的になりうる遺伝子を順位づける

2 万遺伝子から出発して、3 段階で絞り込みます。段階ごとに問い方も表示も変えます。
同じやり方を全段階に使うのは無駄で、しかも精度が落ちます（下の実測を参照）。

| 段階 | する事 | 規模 | 表示 | 1 呼び出し |
|---|---|---|---|---|
| **0** | 二値スクリーニング | 20,000 → 1,000 | 記号のみ | 0.31 秒 |
| **1** | 選択式（群 5 ＋その他） | 1,000 → 100 | 記号＋蛋白質名 | 1.20 秒 |
| **2** | 総当たり（両方向） | 100 → 順位 | 記号＋蛋白質名 | 0.57 秒 |

2 万遺伝子で通して約 4 時間です。段階 0 が 1.7 時間を占めます。
**全遺伝子に 1 回ずつ触れるのは段階 0 だけ**で、ここだけが候補数に比例します。

## なぜ段階で変えるのか

**段階 0 は捨てるためだけの段階です。** 残す 1,000 個の中の順序はどうでもよく、
本物が 1,000 位以内に入っていればよい。だから最も安い形（記号のみ・1 遺伝子 1 呼び出し）で足ります。

**段階 1・2 は残った候補どうしを比べる段階です。** ここは同族の偽陽性との勝負になり、
記号だけでは区別できません。UniProt の蛋白質名を足すと、50 遺伝子の総当たりで
正解の順位が [1, 3] から [1, 2] に上がります（下の実測）。
HGNC 正式名は同族でほぼ同型（`solute carrier family N member M`）なので、
足すとプロンプトが伸びるだけで精度は落ちます。

## この構成に至った実測

いずれも qwen3:14b / cystinuria（正解は SLC3A1 と SLC7A9）。

**段階 0 — 遺伝子名をプロンプトの末尾に置くと 1.8 倍速い**（1,000 遺伝子）

| 配置 | 1 回 | 正解の位置（1,000 個中） | 20,000 件 |
|---|---|---|---|
| 遺伝子名が中間 | 0.56 秒 | SLC7A9 1 位 / SLC3A1 3 位 | 3.1 時間 |
| **遺伝子名が末尾** | **0.31 秒** | **SLC3A1 1 位 / SLC7A9 2 位** | **1.7 時間** |

末尾に置くと、指示文・few-shot・疾患名・設問がすべて固定の前置きになり、
KV キャッシュがそのまま再利用されます。速いうえに順位も良くなりました。

**段階 1 — 蛋白質名を足すと同族に勝てる**（群 5、正解 1 個＋同族 SLC 囮 4 個、各 20 回）

| 表示 | SLC3A1 | SLC7A9 | 合計 | 1 回 |
|---|---|---|---|---|
| 記号のみ | 8/20 | 17/20 | 25/40 | 0.67 秒 |
| **記号＋蛋白質名** | **16/20** | 18/20 | **34/40** | 1.20 秒 |
| 記号＋HGNC 名＋蛋白質名 | 14/20 | 17/20 | 31/40 | 1.73 秒 |

**段階 2 — 総当たりでも同じ**（50 遺伝子・全て OT 関連上位という最難条件）

| 表示 | 時間 | 正解の順位 | 上位 5 |
|---|---|---|---|
| 記号のみ | 12.5 分 | [1, **3**] | SLC7A9, **SLC22A12**, SLC3A1, SLC12A1, SLC14A2 |
| **記号＋蛋白質名** | 23.2 分 | **[1, 2]** | **SLC3A1, SLC7A9**, SLC22A12, SLC12A1, SLC12A3 |

## 限界

**同族で機能の近い遺伝子は、ここまでやっても混ざります。** 上の総当たりでも 3 位は
SLC22A12（尿酸トランスポーター、cystinuria とは無関係）です。上位 2 つは正解でも、
3 位以下は同族の偽陽性が占めます。**上位数個を仮説の種として読む道具**であって、
順位表をそのまま信じる道具ではありません。


## 0. 準備


In [ ]:
import csv, itertools, json, math, os, random, statistics, string, time
import urllib.error, urllib.request
from collections import Counter, defaultdict

try:
    import pandas as pd
    HAVE_PANDAS = True
except ImportError:
    HAVE_PANDAS = False
    print("pandas がありません:  pip install pandas （表は簡易表示になります）")

OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
print("Ollama:", OLLAMA_HOST)


# ===== 保存先 =====
# 段階ごとに結果を残す。1 回の実行につき outputs/日付-時刻/ を 1 つ作り、
# その中に段階 0/1/2 の生データと run.json（設定と実測値）を置く。
# 段階 2 まで 3〜4 時間かかるので、途中で落ちても前の段階をやり直さずに済むようにする。
OUTPUT_ROOT = os.environ.get("GDP_OUTPUT_ROOT", "outputs")
RUN_DIR = None            # ① で作る


def new_run_dir(root=OUTPUT_ROOT):
    """outputs/YYYYMMDD-HHMMSS/ を作って返す。同じ秒に 2 回走っても衝突しない。"""
    base = os.path.join(root, time.strftime("%Y%m%d-%H%M%S"))
    d, k = base, 1
    while os.path.exists(d):
        d = f"{base}_{k}"
        k += 1
    os.makedirs(d)
    return d


def save_json(name, obj):
    """RUN_DIR に JSON を書く。RUN_DIR が無ければ何もしない。"""
    if not RUN_DIR:
        return None
    p = os.path.join(RUN_DIR, name)
    with open(p, "w") as fh:
        json.dump(obj, fh, ensure_ascii=False, indent=1)
    print(f"  保存: {p}")
    return p


def save_rows(name, header, rows):
    """RUN_DIR に TSV を書く。rows は header と同じ並びの列の列。"""
    if not RUN_DIR:
        return None
    p = os.path.join(RUN_DIR, name)
    with open(p, "w", newline="") as fh:
        w = csv.writer(fh, delimiter="\t")
        w.writerow(header)
        w.writerows(rows)
    print(f"  保存: {p}（{len(rows)} 行）")
    return p


def record_meta(**kw):
    """run.json を少しずつ更新する。段階が終わるたびに呼ぶ。

    最後にまとめて書くと、途中で落ちたときに何も残らない。"""
    if not RUN_DIR:
        return
    p = os.path.join(RUN_DIR, "run.json")
    meta = {}
    if os.path.exists(p):
        with open(p) as fh:
            meta = json.load(fh)
    meta.update(kw)
    with open(p, "w") as fh:
        json.dump(meta, fh, ensure_ascii=False, indent=1)


## ① 疾患名と候補リスト


In [ ]:
# ===== 疾患名。プロンプトにこのまま入る =====
DISEASE = "cystinuria"

# ===== 候補遺伝子のファイル =====
#   1 行 1 遺伝子、「記号<TAB>HGNC正式名<TAB>UniProtタンパク質名」。
#   名前はどちらも ④ の USE_* で表示を切り替える。列が無くても動く。
#   genelist_cystinuria_5000.txt      SLC3A1 / SLC7A9 が正解。答え合わせがしやすい
#   genelist_schizophrenia_5000.txt   多遺伝子性。正解が一つに定まらない
GENE_FILE = "genelist01_all.txt"

# ===== 答え合わせ用（任意）=====
#   ここに書いた遺伝子が最後に何位に来たかを ⑧ が表示します。
KNOWN_ANSWERS = ["SLC3A1", "SLC7A9"]


def load_genes(path):
    """「記号<TAB>HGNC正式名<TAB>UniProtタンパク質名」を読む。

    名前の列は無くてもよい（その場合は記号だけを使う）。
    # で始まる行と空行は読み飛ばし、重複は順序を保って落とす。"""
    seen, genes, names, proteins, dupes = set(), [], {}, {}, 0
    with open(path) as f:
        for line in f:
            line = line.rstrip("\n")
            if not line.strip() or line.startswith("#"):
                continue
            parts = line.split("\t")
            g = parts[0].strip()
            if not g or g in seen:
                dupes += g in seen
                continue
            seen.add(g)
            genes.append(g)
            if len(parts) > 1 and parts[1].strip():
                names[g] = parts[1].strip()
            if len(parts) > 2 and parts[2].strip():
                proteins[g] = parts[2].strip()
    return genes, names, proteins, dupes


GENES, GENE_NAMES, PROTEIN_NAMES, n_dupes = load_genes(GENE_FILE)
print(f"疾患  : {DISEASE}")
print(f"候補  : {len(GENES)} 遺伝子" + (f"（重複 {n_dupes} 件を除外）" if n_dupes else ""))
print(f"HGNC 正式名     : {len(GENE_NAMES)}/{len(GENES)} 件")
print(f"UniProt 蛋白質名: {len(PROTEIN_NAMES)}/{len(GENES)} 件")
missing = [g for g in KNOWN_ANSWERS if g not in GENES]
if missing:
    print(f"⚠ 答え合わせ用の {missing} が候補に入っていません")
else:
    print(f"答え合わせ: {KNOWN_ANSWERS}（いずれも候補に含まれる）")


# ===== この実行の保存先 =====
RUN_DIR = new_run_dir()
print(f"保存先: {RUN_DIR}/")
record_meta(started=time.strftime("%Y-%m-%d %H:%M:%S"), disease=DISEASE,
            gene_file=GENE_FILE, n_genes=len(GENES),
            n_protein_names=len(PROTEIN_NAMES), known_answers=KNOWN_ANSWERS)


## ② モデルを選ぶ

ローカルの Ollama に入っているモデルを一覧します。`MODEL` に既定を入れてください。
空のままなら一覧の先頭を使います。

**段階ごとに別のモデルを指定できます。**段階によって要求が違うためです。

段階 0 は 2 万件を 1 件ずつ捌く場所で、要るのは速さと「本物を落とさない」ことだけです。
どの偽陽性を通すかは段階 1・2 が決めるので、ここに賢いモデルは要りません。
段階 2 は対象が 100 個まで減っており、ここが最終順位を決めるので 1 件あたりのコストを
払えます。`SCREEN_MODEL` / `RANK_MODEL` / `RR_MODEL` を空にすると `MODEL` を使います。

⚠ 段階ごとに違うモデルを使うと、VRAM に 2 つ載らないとき呼び出しのたびに載せ替えが
起きます。実測で 1 件 0.3 秒が 30 秒になりました。段階の切れ目で `unload()` を呼んで
ください（Ollama サーバー自体は止めません）。


In [3]:
def list_local_models():
    """Ollama に入っているモデル名。埋め込み専用モデルは logprobs を返さないので外す。"""
    try:
        with urllib.request.urlopen(OLLAMA_HOST + "/api/tags", timeout=10) as r:
            models = json.loads(r.read().decode()).get("models", [])
    except urllib.error.URLError as e:
        print(f"Ollama に接続できません（{e.reason}）。`ollama serve` は動いていますか。")
        return []
    skip = ("embed", "bge-", "e5-", "gte-")
    return [{"name": m["name"],
             "params": m.get("details", {}).get("parameter_size", "?"),
             "size_gb": round(m.get("size", 0) / 1e9, 1)}
            for m in models if not any(k in m["name"].lower() for k in skip)]


AVAILABLE = list_local_models()
for i, m in enumerate(AVAILABLE):
    print(f"  [{i}] {m['name']:<30}{m['params']:>8}  {m['size_gb']}GB")
if not AVAILABLE:
    print("  （使えるモデルが見つかりません）")


  [0] gemma3:27b                       27.4B  17.4GB
  [1] qwen2.5:7b                        7.6B  4.7GB
  [2] qwen3:14b                        14.8B  9.3GB
  [3] cniongolo/biomistral:latest         7B  4.4GB
  [4] gemma3:4b-it-qat                  4.3B  4.0GB
  [5] deepseek-r1:8b                    8.2B  5.2GB
  [6] qwen3:8b                          8.2B  5.2GB
  [7] qwen2.5:14b                      14.8B  9.0GB
  [8] llama3.1:latest                   8.0B  4.9GB


In [ ]:
MODEL = ""        # ← 既定のモデル。例: "qwen3:14b"。空なら一覧の先頭

# ===== 段階ごとのモデル。空なら MODEL を使う =====
# 実測（cystinuria 1,000 件、段階 0 の Yes/No）:
#   qwen3:14b    0.32 秒/件  正解 1 位・2 位  境界との差 +22
#   llama3.1:8b  0.17 秒/件  正解 1 位・3 位  境界との差 +3.4
# 2 万件なら 1.79 時間 → 0.96 時間。統合失調症 1,000 件でも再現率は同等だった。
# ただし上位 50 の一致は 32% しかなく、両者は別の基準で選んでいる。順位を信じて
# よいのは段階 1・2 のモデルだけで、段階 0 の出力は「集合」として扱うこと。
SCREEN_MODEL = "qwen3:8b"     # 段階 0（Yes/No）
RANK_MODEL   = "qwen3:14b"    # 段階 1（群 5 の選択式）
RR_MODEL     = "qwen3:14b"    # 段階 2（総当たり）

if not MODEL and AVAILABLE:
    MODEL = AVAILABLE[0]["name"]
SCREEN_MODEL = SCREEN_MODEL or MODEL
RANK_MODEL = RANK_MODEL or MODEL
RR_MODEL = RR_MODEL or MODEL

STAGE_MODELS = [("段階 0  二値スクリーニング", SCREEN_MODEL),
                ("段階 1  選択式（群 5）", RANK_MODEL),
                ("段階 2  総当たり", RR_MODEL)]


def unload(model):
    """そのモデルだけを VRAM から降ろす。`ollama stop <model>` と同じ。

    Ollama サーバー本体は止めない。次に使えば自動で読み直される。"""
    payload = {"model": model, "prompt": "", "keep_alive": 0}
    req = urllib.request.Request(
        OLLAMA_HOST + "/api/generate", data=json.dumps(payload).encode(),
        headers={"Content-Type": "application/json"}, method="POST")
    try:
        with urllib.request.urlopen(req, timeout=60):
            print(f"降ろしました: {model}")
    except urllib.error.URLError as e:
        print(f"降ろせませんでした: {model} ({e})")


_sizes = {m["name"]: m["size_gb"] for m in AVAILABLE}   # AVAILABLE は GB 済み
for _label, _m in STAGE_MODELS:
    if _m and AVAILABLE and _m not in _sizes:
        print(f"⚠ {_m} は一覧にありません。`ollama pull {_m}` が要るかもしれません。")
print(f"  {'段階':<26}{'モデル':<28}{'サイズ':>9}")
for _label, _m in STAGE_MODELS:
    print(f"  {_label:<26}{_m or '★未選択':<28}{_sizes.get(_m, 0):>8.1f}GB")

_distinct = {m for _, m in STAGE_MODELS if m}
if len(_distinct) > 1:
    print(f"\n  段階ごとに {len(_distinct)} 個のモデルを使います"
          f"（合計 {sum(_sizes.get(m, 0) for m in _distinct):.1f}GB）。")
    print("  同時に常駐して VRAM を超えると、呼び出しのたびに載せ替えが起きて")
    print("  30 倍以上遅くなります。段階が終わったら unload() で降ろしてください。")

record_meta(models={"stage0": SCREEN_MODEL, "stage1": RANK_MODEL, "stage2": RR_MODEL})


### モデルとのやりとり

次の 1 トークンの対数確率だけを読みます。文章は生成させません。
段階 0 は Yes / No の、段階 1・2 は A / B / C… の対数確率を、いずれもこの関数から取ります。


In [5]:
def first_token_logprobs(prompt, model, top_logprobs=20, no_think=True):
    """次の 1 トークンの (生成トークン, {トークン: 対数確率})。Ollama の上限は 20。

    no_think は思考モデル対策。対応していないモデルに送ると弾かれるので、
    その場合は外して再送する。"""
    payload = {"model": model, "prompt": prompt, "stream": False,
               "options": {"temperature": 0, "num_predict": 1},
               "logprobs": True, "top_logprobs": min(top_logprobs, 20)}
    if no_think:
        payload["think"] = False
    req = urllib.request.Request(
        OLLAMA_HOST + "/api/generate", data=json.dumps(payload).encode(),
        headers={"Content-Type": "application/json"}, method="POST")
    try:
        with urllib.request.urlopen(req, timeout=120) as r:
            data = json.loads(r.read().decode())
    except urllib.error.HTTPError:
        if not no_think:
            raise
        return first_token_logprobs(prompt, model, top_logprobs, no_think=False)
    if "error" in data:
        raise RuntimeError(str(data["error"])[:200])
    lp = (data.get("logprobs") or [None])[0]
    if not lp:
        raise RuntimeError("logprobs が返りません。Ollama v0.12.11 以降が必要です。")

    out = {}
    tok, val = lp.get("token"), lp.get("logprob")
    if tok is not None and val is not None:
        out[tok.strip()] = val
    for alt in (lp.get("top_logprobs") or []):
        if not isinstance(alt, dict):
            continue
        t, v = alt.get("token"), alt.get("logprob")
        if t is None:                       # {token: logprob} 形式のことがある
            for k, vv in alt.items():
                out.setdefault(str(k).strip(), vv)
        elif v is not None:
            out.setdefault(str(t).strip(), v)
    return tok, out


## ③ 段階 0 — 二値スクリーニング

候補を 1 個ずつ独立に見て、「この遺伝子を動かしてこの疾患を治す**既知の機序**があるか」を
Yes / No で聞き、`logP(Yes) − logP(No)` を連続値として取ります。
群にして比べないので、**関係する遺伝子が 1 つも入っていない群で偽の勝者が出る**という
選択式の構造的な問題がここでは起きません。

```
Answer Yes or No.

Disease: Cystic fibrosis
Is there a known mechanism by which modulating this gene would treat this disease?
Gene: CFTR
Answer: Yes

Disease: Cystic fibrosis
Is there a known mechanism by which modulating this gene would treat this disease?
Gene: APOE
Answer: No

Disease: <対象疾患>
Is there a known mechanism by which modulating this gene would treat this disease?
Gene: <遺伝子>
Answer:
```

**遺伝子名が最後に来ている**のが要点です。指示文から設問までが全候補で共通なので、
その部分の計算が使い回されます。遺伝子名を途中に置く書き方より 1.8 倍速い。

### 「機序があるか」と聞く

同じ二値でも問い方で大きく変わります（正解 − 同族偽陽性の差、大きいほど良い）。

| 設問 | 分離 |
|---|---|
| `Is this gene a plausible therapeutic target for this disease?` | 1.85 |
| `Could modulating this gene plausibly treat this disease?` | 2.46 |
| **`Is there a known mechanism by which modulating this gene would treat this disease?`** | **8.54** |

なおこの文言は**段階 1 では使えません**。選択式の設問をこれに差し替えると、
関係する遺伝子が無い群での棄却率が 5/20 から 0/20 に落ちます。
段階ごとに最良の文言が違うので、揃えないでください。

### 進捗の見え方

この段階だけが候補数に比例します（2 万件で 1.7 時間）。止まっているのか進んでいるのか
分からないと困るので、候補を `SCREEN_CYCLES` 等分し、1 サイクルごとに 1 行出します。

```
Ollama に常駐中: qwen3:14b (10GB)
段階 0  20000 遺伝子 x 1 呼び出し  （20 サイクル、1 サイクル 1000 遺伝子）
    1/20 [#...................]   1000/20000    0.31秒/件  経過   5.2分  残り  98.1分  >0: 58 (5.8%)  首位 SLC3A1 +25.0
    2/20 [##..................]   2000/20000    0.31秒/件  経過  10.4分  残り  93.0分  >0: 121 (6.1%)  首位 SLC3A1 +25.0
```

行を上書きせずに積むのは、あとから「どのあたりで遅くなったか」を読み返せるようにするためです。

`秒/件` は**通算ではなくそのサイクルだけ**の値です。通算にすると、遅い区間を抜けても
数字が下がりきらず、復帰したことが見えません（実測で最後まで 8.1 秒と表示され続けたが、
実際は途中から 0.3 秒に戻っていた）。残り時間もこの直近の速度から計算します。

### 遅いときは載せ替えを疑う

**実測でこれが起きました。** 200 遺伝子のうち最初の 50 個に 24 分、残り 150 個は 3 分。
サーバーログを見ると 13〜16 秒ごとに 2 つのモデルが交互にロードし直されていました。

```
18:32:25  offloaded 57/63 layers  (gemma3:27b)   VRAM残 3.0 GiB
18:32:38  offloaded 41/41 layers  (qwen3:14b)    VRAM残 8.8 GiB
18:32:54  offloaded 57/63 layers  (gemma3:27b)   VRAM残 3.0 GiB
```

VRAM 17.8 GiB に gemma3:27b（約 15GB）と qwen3:14b（10GB）は同居できません。
片方を呼ぶともう片方が追い出され、次の呼び出しで再ロードが走ります。
**1 件 0.3 秒の処理が 30 秒になります。**

推論そのものは速いままです（`prompt eval time = 329ms / 19 tokens`、
前置き 102 トークンはキャッシュから再利用）。遅いのは計算ではなくロードです。

そのため実行前に常駐モデルを表示し、2 つ以上あれば警告します。
走行中も、あるサイクルが最速の 3 倍以上遅ければその場で指摘します。

### 絞り方

閾値の絶対値は疾患ごとにずれるので、**順位で切ります**（上位 `SCREEN_KEEP` 個）。
実測（cystinuria・1,000 遺伝子）では `logit差 > 0` の通過率が 6.2%、
正解 2 つはどちらも 1 位・2 位でした。2 万件なら 1,200 個前後が 0 を超える計算です。


In [ ]:
SCREEN_KEEP = 1000        # 段階 1 へ送る数。閾値ではなく順位で切る
SCREEN_MIN_MARGIN = None  # 併用する下限（例 0.0）。None なら順位だけで切る
SCREEN_CYCLES = 20        # 進捗を何サイクルに分けて表示するか

SCREEN_QUESTION = ("Is there a known mechanism by which modulating this gene "
                   "would treat this disease?")
# few-shot は承認薬で裏の取れたペア。CFTR は ivacaftor の標的、APOE は嚢胞性線維症と無関係。
SCREEN_SHOTS = [("Cystic fibrosis", "CFTR", "Yes"), ("Cystic fibrosis", "APOE", "No")]


def screen_prefix(disease, question=SCREEN_QUESTION):
    """全候補で共通の前置き。遺伝子名はこの後ろに付く。

    遺伝子名を末尾に置くのは表示の都合ではなく速度のためで、ここまでが
    全呼び出しで同一なので KV キャッシュがそのまま効く。設問を遺伝子の
    後ろに置く書き方だと、遺伝子が変わるたびに設問文を計算し直すことになる。"""
    shots = "".join(f"Disease: {d}\n{question}\nGene: {g}\nAnswer: {a}\n\n"
                    for d, g, a in SCREEN_SHOTS)
    return f"Answer Yes or No.\n\n{shots}Disease: {disease}\n{question}\nGene: "


def screen_margin(gene, prefix, model):
    """logP(Yes) − logP(No)。上位 20 に入らなかった側は -99 として扱う。

    差を取るのは、Yes と No の絶対値が疾患名の長さや語調で上下するため。
    差にすると、その共通成分が消える。"""
    _, raw = first_token_logprobs(prefix + gene + "\nAnswer:", model)
    return raw.get("Yes", -99.0) - raw.get("No", -99.0)


def _bar(done, total, width=20):
    n = int(width * done / total) if total else 0
    return "[" + "#" * n + "." * (width - n) + "]"


def resident_models():
    """いま Ollama に載っているモデル。[(名前, GB), ...]。

    このマシンの VRAM は有限で、大きいモデルが 2 つは載らない。載らないと
    呼び出しのたびに載せ替えが起きて、1 件 0.3 秒の処理が 30 秒になる。
    実行前に何が居るかを見せて、`ollama stop <model>` で降ろす判断ができるようにする。"""
    try:
        with urllib.request.urlopen(OLLAMA_HOST + "/api/ps", timeout=10) as r:
            ms = json.loads(r.read().decode()).get("models", [])
    except urllib.error.URLError:
        return []
    return [(m.get("name", "?"), m.get("size", 0) / 1e9) for m in ms]


def screen(disease, genes, model, keep=SCREEN_KEEP, min_margin=SCREEN_MIN_MARGIN,
           cycles=SCREEN_CYCLES):
    """全候補を 1 個ずつ採点し、上位 keep 個を返す。(生き残り, {遺伝子: 差})。

    候補数を cycles 等分し、1 サイクル終わるごとに 1 行出す。2 万件で 1.7 時間
    かかる段階なので、動いているのか止まっているのかが分からないと困る。
    行を上書きせずに積むのは、あとから「どのあたりで遅くなったか」を
    読み返せるようにするため。

    速度は**そのサイクルだけ**の秒/件で出す。通算にすると、遅い区間を抜けても
    数字が下がりきらず、復帰したことが見えない（実測で最後まで 8.1 秒と
    表示され続けたが、実際は途中から 0.3 秒に戻っていた）。"""
    prefix = screen_prefix(disease)
    total = len(genes)
    step = max(1, -(-total // max(1, cycles)))      # 1 サイクルあたりの遺伝子数
    margins, failures = {}, 0
    t0 = time.time()

    res = resident_models()
    if res:
        print("Ollama に常駐中: "
              + "、".join(f"{n} ({g:.0f}GB)" for n, g in res))
        if len(res) > 1:
            print("  ⚠ 2 つ以上載っています。VRAM が足りないと呼び出しのたびに"
                  "載せ替えが起き、30 倍以上遅くなります。\n"
                  "    使わないものは `ollama stop <名前>` で降ろしてください。")
    print(f"段階 0  {total} 遺伝子 x 1 呼び出し  モデル {model}  "
          f"（{cycles} サイクル、1 サイクル {step} 遺伝子）", flush=True)

    t_cycle, best_rate = t0, None
    for i, g in enumerate(genes, 1):
        try:
            margins[g] = screen_margin(g, prefix, model)
        except Exception as e:
            failures += 1
            if failures <= 3:
                print(f"    {g}: 失敗 {type(e).__name__}: {e}")

        if i % step and i != total:
            continue
        # --- 1 サイクル終了 ---
        now = time.time()
        cyc = -(-i // step)
        done_here = step if i % step == 0 else i % step
        rate = (now - t_cycle) / done_here             # このサイクルだけの秒/件
        t_cycle = now
        best_rate = rate if best_rate is None else min(best_rate, rate)
        eta = (total - i) * rate
        over0 = sum(1 for v in margins.values() if v > 0)
        best = max(margins, key=margins.get) if margins else None
        print(f"  {cyc:>3}/{cycles} {_bar(i, total)} {i:>6}/{total}"
              f"  {rate:>6.2f}秒/件  経過 {(now - t0) / 60:>5.1f}分"
              f"  残り {eta / 60:>5.1f}分"
              f"  >0: {over0} ({over0 / len(margins) * 100:.1f}%)"
              + (f"  首位 {best} {margins[best]:+.1f}" if best else ""), flush=True)
        if best_rate and rate > 3 * best_rate and rate > 2:
            print(f"    ⚠ このサイクルは最速の {rate / best_rate:.0f} 倍遅い。"
                  "別のモデルとの載せ替えが起きていないか `ollama ps` で確認を。",
                  flush=True)

    ranked = sorted(margins, key=lambda g: -margins[g])
    if min_margin is not None:
        ranked = [g for g in ranked if margins[g] > min_margin]
    survivors = ranked[:keep] if keep else ranked
    el = time.time() - t0
    print(f"\n  {total} → {len(survivors)}  "
          f"{el / 60:.1f}分（通算 {el / max(total, 1):.2f}秒/件、"
          f"最速サイクル {best_rate:.2f}秒/件、失敗 {failures}）")
    if survivors:
        print(f"  通過の下限: {margins[survivors[-1]]:+.2f}"
              f"（{len(survivors)} 位）／上限 {margins[survivors[0]]:+.2f}")
    return survivors, margins


In [ ]:
SURVIVORS, SCREEN_MARGIN = list(GENES), {}

if SCREEN_MODEL and GENES:
    if len(GENES) > SCREEN_KEEP:
        SURVIVORS, SCREEN_MARGIN = screen(DISEASE, GENES, SCREEN_MODEL)
        print(f"\n  上位 15: {SURVIVORS[:15]}")
        for g in KNOWN_ANSWERS:
            if g not in SCREEN_MARGIN:
                print(f"  {g:<10} 採点されていません")
            else:
                pos = sorted(SCREEN_MARGIN, key=lambda x: -SCREEN_MARGIN[x]).index(g) + 1
                mark = "通過" if g in SURVIVORS else "⚠ 脱落"
                print(f"  {g:<10} {pos} 位 / {len(SCREEN_MARGIN)}"
                      f"  (logit差 {SCREEN_MARGIN[g]:+.2f})  {mark}")
    else:
        print(f"候補 {len(GENES)} 個は SCREEN_KEEP({SCREEN_KEEP}) 以下なので"
              f"スクリーニングを飛ばします。")
else:
    print("モデル未選択のため実行しません。")

# ===== 段階 0 の保存 =====
# 全候補の logit 差を残す。ここを取り直すのが一番高くつく（19,297 件で 1 時間）ので、
# 通過した 1,000 個だけでなく落ちた分も書く。閾値を変えて切り直せるようにするため。
if SCREEN_MARGIN:
    _ranked = sorted(SCREEN_MARGIN, key=lambda g: -SCREEN_MARGIN[g])
    _pass = set(SURVIVORS)
    save_rows("stage0_screen.tsv",
              ["rank", "gene", "protein_name", "logit_margin", "passed"],
              [[i, g, PROTEIN_NAMES.get(g, ""), round(SCREEN_MARGIN[g], 4),
                int(g in _pass)] for i, g in enumerate(_ranked, 1)])
    record_meta(stage0={"model": SCREEN_MODEL, "scored": len(SCREEN_MARGIN),
                        "kept": len(SURVIVORS), "keep_setting": SCREEN_KEEP,
                        "min_margin_setting": SCREEN_MIN_MARGIN,
                        "cut_margin": round(SCREEN_MARGIN[SURVIVORS[-1]], 4) if SURVIVORS else None,
                        "top_margin": round(SCREEN_MARGIN[_ranked[0]], 4)})


## ④ プロンプト

モデルには **1 文字のラベルだけ**を答えさせ、遺伝子記号そのものは生成させません。
記号を書かせると GPR52 と GPR56 のような似た記号を取り違えます。

### 「関連が強い遺伝子」ではなく「治療標的」を聞く

問い方でまったく違う答えが返ります。統合失調症で、無作為な 25 個を相手に 20 回試した実測:

| 遺伝子 | 「最も強く関連する」 | **「治療標的として最も重要」** | |
|---|---|---|---|
| CHRM4 | 4/20 | **19/20** | キサノメリンの標的 |
| DRD2 | 15/20 | **20/20** | 抗精神病薬 48/50 剤の標的 |
| HTR2A | 17/20 | **20/20** | 抗精神病薬 37 剤の標的 |
| TCF4 | 16/20 | **6/20** | 薬が結合しない転写因子 |
| ZNF804B | 20/20 | **18/20** | 証拠の乏しい同族遺伝子 |

前者では GWAS 文献に頻出する遺伝子が勝ち、薬が結合する遺伝子が沈みます。
実際、5,000 遺伝子中 DRD2 が 106 位まで落ちたことがありました。

### few-shot の正解をどの文字に置くかが結果を変える

無関係な遺伝子だけの群を 60 回投げて、勝ったラベルを数えた結果です（一様なら各 2.3 回）。

| few-shot の正解 | B が勝った回数 | A が勝った回数 |
|---|---|---|
| B と B | **34/60** | 0 |
| B と C（現在） | 0/60 | 16/60 |

2 問とも B に置くと、無関係な群の 6 割で B が勝ちます — 内容ではなくラベルが勝者を決めていました。
B と C に散らすと消えます。先頭 A への偏りは残りますが、⑤ がラベル割り当てを毎回振り直すので、
どの遺伝子も 1/26 の確率で A に座るだけになり、系統誤差ではなく分散になります。

（同じ群をラベル順を変えて複数回投げる方法も試しましたが、**呼び出し予算を固定すると差が出ません**。
1・2・3 ローテーションのいずれでも承認薬の標的が 1〜6 位を独占し、結果は完全に同一でした。）

### 記号だけでなく正式名も見せる

記号だけを並べると、モデルは**同族の記号に引かれて中身を見なくなります**。cystinuria で実測した例：

同族の SLC を 25 個並べ、そこに正解を 1 個混ぜて 20 回ずつ出題した結果です。

| 条件 | SLC3A1 | SLC7A9 | 合計 | プロンプト長 |
|---|---|---|---|---|
| 記号のみ | 1/20 | 6/20 | 7/40 (18%) | 594 字 |
| **記号＋正式名（現在）** | 2/20 | **11/20** | **13/40 (33%)** | 1,812 字 |
| 記号＋別名 | 3/20 | 9/20 | 12/40 (30%) | 1,219 字 |

`solute carrier family 7 member 9` と `solute carrier family 7 member 11` は文字列としてほぼ同じなので
効かないと予想していましたが、外れました。18% → 33% と約 2 倍になります。

**代償はプロンプト長 3 倍、つまり所要時間も約 3 倍です。** 所要時間はプロンプト長にほぼ比例します
（実測: 4 択 0.60 秒 / 133 トークン、26 択 1.85 秒 / 279 トークン）。
`USE_GENE_NAMES = False` にすれば記号のみに戻せます。

**それでも 33% です。** 同族の中では 3 回に 2 回間違えます。ファミリーまでは絞れても、
その中の順位は中身を反映していないと考えてください。


In [8]:
LETTERS = list(string.ascii_uppercase)     # A..Z
N_PER_ROUND = 5                            # 1 群の遺伝子数。実測で 5 が最良（上限は 26）
USE_GENE_NAMES = False                     # HGNC 正式名を併記するか。実測では足すと悪くなる
USE_PROTEIN_NAMES = True                   # UniProt の蛋白質推奨名を併記する。同族の区別に効く
USE_EXIT_OPTION = True                     # 「その他」を必ず選択肢に入れる
EXIT_TEXT = "Other / none of the above"    # その選択肢の文言

INSTRUCTION = "Answer with a single letter only.\n\n"
QUESTION = "Most important therapeutic target:"


def label_for(gene):
    """選択肢 1 行分の表示。記号のみが既定で、名前は任意で添える。

    UniProt の推奨名は機能を書いている（SLC3A1 = Amino acid transporter heavy chain、
    SLC7A9 = b(0,+)-type amino acid transporter 1）ので、同族の区別に効く。
    実測で群 5 の正解率が 25/40 から 34/40 に上がる。

    HGNC 正式名は同族でほぼ同一（solute carrier family N member M）で情報を足さない。
    併記すると 31/40 に下がり、1 呼び出しも 1.20 秒から 1.73 秒に伸びる。
    両方を有効にすると ; で連結する。"""
    parts = []
    if USE_GENE_NAMES and GENE_NAMES.get(gene):
        parts.append(GENE_NAMES[gene])
    if USE_PROTEIN_NAMES and PROTEIN_NAMES.get(gene):
        parts.append(PROTEIN_NAMES[gene])
    return f"{gene} ({'; '.join(parts)})" if parts else gene


# few-shot は承認薬とその標的で揃える。どちらも ChEMBL の作用機序で裏を取ったペア。
#   Cystic fibrosis / CFTR      ivacaftor (ACTIVATOR)
#   Rheumatoid arthritis / TNF  adalimumab (INHIBITOR)
# 正解は B と C。同じ文字に寄せない（理由は上の説明を参照）。
# 本番と同じ体裁にするため、few-shot の選択肢にも正式名を入れる。
_FEWSHOT_PROTEINS = {
    "HBB": "Hemoglobin subunit beta",
    "CFTR": "Cystic fibrosis transmembrane conductance regulator",
    "GPR52": "G-protein coupled receptor 52",
    "APOE": "Apolipoprotein E",
    "TNF": "Tumor necrosis factor",
    "GPR56": "Adhesion G-protein coupled receptor G1",
}

_FEWSHOT_NAMES = {
    "HBB": "hemoglobin subunit beta",
    "CFTR": "CF transmembrane conductance regulator",
    "GPR52": "G protein-coupled receptor 52",
    "APOE": "apolipoprotein E",
    "TNF": "tumor necrosis factor",
    "GPR56": "adhesion G protein-coupled receptor G1",
}


def _fewshot_block(disease, genes, answer):
    """few-shot も本番と同じ体裁にする。「その他」を入れるなら例にも入れる。

    形式が食い違うと、モデルが 1 文字ではなく散文を書き始めることがある。"""
    items = list(genes) + ([None] if USE_EXIT_OPTION else [])
    lines = []
    for lab, g in zip(LETTERS, items):
        if g is None:
            lines.append(f"{lab}. {EXIT_TEXT}")
            continue
        parts = []
        if USE_GENE_NAMES and _FEWSHOT_NAMES.get(g):
            parts.append(_FEWSHOT_NAMES[g])
        if USE_PROTEIN_NAMES and _FEWSHOT_PROTEINS.get(g):
            parts.append(_FEWSHOT_PROTEINS[g])
        lines.append(f"{lab}. " + (f"{g} ({'; '.join(parts)})" if parts else g))
    return (f"Disease: {disease}\n{QUESTION}\n" + "\n".join(lines)
            + f"\nAnswer: {answer}\n\n")


FEWSHOT = (
    _fewshot_block("Cystic fibrosis", ["HBB", "CFTR", "GPR52", "APOE"], "B")
    + _fewshot_block("Rheumatoid arthritis", ["CFTR", "HBB", "TNF", "GPR56"], "C")
)


def build_prompt(disease, genes, exit_option=None):
    """1 群 → プロンプトと {ラベル: 遺伝子記号} の対応表。

    対応表が返すのは**記号**で、表示用の文字列ではない。ラベルは呼ぶたびに違う
    遺伝子を指すので、「A が正解」ではなく「A が指していたもの」をこの表から読み戻す。

    「その他」を足すと、その対応値は None になる。既定は USE_EXIT_OPTION に従うが、
    段階 2 の 1 対 1 では exit_option=False で切る。2 個から選ぶ場面で逃げ道を残すと、
    どちらが強いかという情報そのものが取れない。
    候補 5,000 個を 5 個ずつ配れば 1 周で 1,000 群できる。そのほとんどには
    関係する遺伝子が 1 つも入っていないので、逃げ道が無いと 1,000 個の
    偽の勝者が生まれる。"""
    if exit_option is None:
        exit_option = USE_EXIT_OPTION
    items = list(genes) + ([None] if exit_option else [])
    mapping = dict(zip(LETTERS, items))
    body = "\n".join(
        f"{lab}. " + (EXIT_TEXT if g is None else label_for(g))
        for lab, g in mapping.items())
    prompt = (INSTRUCTION + FEWSHOT + f"Disease: {disease}\n{QUESTION}\n"
              + body + "\nAnswer:")
    return prompt, mapping


print(build_prompt(DISEASE, GENES[:N_PER_ROUND])[0][:1200], "...")


Answer with a single letter only.

Disease: Cystic fibrosis
Most important therapeutic target:
A. HBB (Hemoglobin subunit beta)
B. CFTR (Cystic fibrosis transmembrane conductance regulator)
C. GPR52 (G-protein coupled receptor 52)
D. APOE (Apolipoprotein E)
E. Other / none of the above
Answer: B

Disease: Rheumatoid arthritis
Most important therapeutic target:
A. CFTR (Cystic fibrosis transmembrane conductance regulator)
B. HBB (Hemoglobin subunit beta)
C. TNF (Tumor necrosis factor)
D. GPR56 (Adhesion G-protein coupled receptor G1)
E. Other / none of the above
Answer: C

Disease: cystinuria
Most important therapeutic target:
A. ABCC5 (ATP-binding cassette sub-family C member 5)
B. ABCG2 (Broad substrate specificity ATP-binding cassette transporter ABCG2)
C. ABHD14A (Protein ABHD14A)
D. ABI2 (Abl interactor 2)
E. ACE (Angiotensin-converting enzyme)
F. Other / none of the above
Answer: ...


## ⑤ 1 群を採点する


In [9]:
def ranker(disease, genes, model, shuffle_labels=True, exit_option=None):
    """遺伝子の一群 → {遺伝子: スコア}。スコアは群の中で正規化した確率。

    ラベルの割り当ては既定でシャッフルする。固定すると、top-20 に入り損ねる位置の
    遺伝子が毎回同じになって偏るため。

    exit_option は「その他」を入れるかどうか。既定は USE_EXIT_OPTION に従う。
    段階 2 の 1 対 1 では False で呼ぶ。

    ラベルが 1 つも返らなかったときは空を返す。全部を下限値で埋めると
    「動いた」ように見えてしまい、壊れていることに気づけない。"""
    genes = list(genes)
    if shuffle_labels:
        random.shuffle(genes)
    prompt, mapping = build_prompt(disease, genes, exit_option=exit_option)
    gen_token, raw = first_token_logprobs(prompt, model)

    labels = list(mapping)
    got = {lab: raw[lab] for lab in labels if lab in raw}
    base = {"generated": gen_token, "n_labels": len(labels),
            "n_missing": len(labels) - len(got)}
    if not got:
        return {**base, "scores": {}, "top": None, "top_prob": None}

    floor = min(got.values()) - 10.0        # 見えなかったラベルは下限に置く
    filled = {lab: got.get(lab, floor) for lab in labels}
    m = max(filled.values())
    exp = {lab: math.exp(v - m) for lab, v in filled.items()}
    z = sum(exp.values())
    prob = {lab: v / z for lab, v in exp.items()}

    top_lab = max(prob, key=prob.get)
    # 「その他」は遺伝子ではないので scores には入れない。入れると ⑥ が
    # None という名前の遺伝子を集計してしまう。確率は別に返す。
    scores = {mapping[lab]: p for lab, p in prob.items() if mapping[lab] is not None}
    return {**base, "scores": scores,
            "top": mapping[top_lab], "top_prob": prob[top_lab],
            "exit_prob": sum(p for lab, p in prob.items() if mapping[lab] is None),
            "picked_exit": mapping[top_lab] is None}


In [ ]:
# 動作確認 — ここで letter が返らなければ、先へ進んでも意味がありません。
# 段階 1・2 のモデル（RANK_MODEL）で確かめる。段階 0 は Yes/No なので別。
if RANK_MODEL:
    random.seed(0)
    probe = ranker(DISEASE, GENES[:N_PER_ROUND], RANK_MODEL)
    print(f"モデル       : {RANK_MODEL}")
    print(f"生成トークン : {probe['generated']!r}")
    print(f"ラベル取得   : {probe['n_labels'] - probe['n_missing']}/{probe['n_labels']}"
          f"（Ollama の上限 20 のため 6 個前後の補完は正常）")
    if not probe["scores"]:
        print("\n❌ ラベルが 1 つも返っていません。")
        print("   ・思考モデルなら think:false が効いているか（生成が <think> なら効いていない）")
        print("   ・④ の INSTRUCTION と FEWSHOT を消していないか")
        print("   ・別のモデルを試す")
    else:
        print(f"1 位         : {probe['top']}  (p={probe['top_prob']:.3f})")
else:
    print("モデル未選択のため実行しません。")


## ⑥ 段階 1 — 選択式で 1,000 から 100 へ

段階 0 を通った候補を、群 5 ＋「その他」で何度も戦わせて上位 100 個に絞ります。
ここからは選択肢に **UniProt の蛋白質名を併記**します（④ の `USE_PROTEIN_NAMES`）。
同族の SLC どうしは記号だけでは区別できず、名前を足すと群 5 の正解率が
25/40 から 34/40 に上がります。

### 山札方式で配る

毎回独立に無作為抽出すると登場回数が二項分布になり、大きくばらつきます。
実測（501 遺伝子・1 群 26・100 ラウンド）では **0〜13 回**、一度も出ない遺伝子も出ました。
13 回出た遺伝子と 1 回の遺伝子の平均を並べても、比べているのは実力ではなく試行回数です。

全遺伝子を 1 つの山に切って上から 26 枚ずつ配り、尽きたら切り直す。
こうすると登場回数の差は**ラウンド数によらず常に最大 1** になります（実測 5〜6 回、未出現ゼロ）。

### 段階的に絞る

一発勝負のトーナメントも試しましたが、10,000 個中 9,615 個が 1 回戦敗退で互いに区別できず、
優勝候補と同じ組になった遺伝子は実力 2 位でも消えました。
こちらは負けても消えません。各段階で全員に平均スコアが付き、下位を切るだけです。

**1 段階で切る比率は 80% 程度までに抑えてください。** 登場 5 回の平均で 90% を落とすと、
運悪く強い群に当たり続けた本物を落とします。


In [ ]:
def balanced_groups(genes, n, rounds, rng):
    """山札方式。全遺伝子を切って上から配り、尽きたら切り直す。

    どの遺伝子も切り直しごとにちょうど 1 回配られるので、登場回数の差は
    ラウンド数によらず最大 1。同一群内の重複だけは飛ばす。"""
    n = min(n, len(genes))
    deck, out = [], []
    for _ in range(rounds):
        group = []
        while len(group) < n:
            if not deck:
                deck = list(genes)
                rng.shuffle(deck)
            picked = None
            for i, g in enumerate(deck):
                if g not in group:          # 同じ群に二度入れない
                    picked = deck.pop(i)
                    break
            if picked is None:              # 山の残りが全部この群にある
                deck = []
                continue
            group.append(picked)
        out.append(group)
    return out


def winnow_preview(n_genes, stages, group_size, sec_per_call):
    """実行前に、各段階の登場回数・呼び出し数・時間を出す。"""
    pool, total = n_genes, 0
    print(f"  {'段階':<5}{'対象':>8}{'ラウンド':>10}{'登場/個':>10}{'時間':>9}")
    for si, (rounds, keep) in enumerate(stages, 1):
        appear = rounds * group_size / pool if pool else 0
        total += rounds
        print(f"  {si:<5}{pool:>8}{rounds:>10}{appear:>10.1f}{rounds * sec_per_call / 60:>8.0f}分")
        if appear < 3:
            print(f"    ⚠ 登場 {appear:.1f} 回では、切る根拠が試行回数の偶然になります。")
        if keep and pool and keep / pool < 0.15:
            print(f"    ⚠ {(1 - keep / pool) * 100:.0f}% を一度に切ります。段階を増やすほうが安全です。")
        pool = min(keep, pool) if keep else pool
    print(f"  {'合計':<5}{'':>8}{total:>10}{'':>10}{total * sec_per_call / 60:>8.0f}分")
    return total


def winnow(disease, genes, model, stages, group_size=26, seed=0,
           progress_every=100, verbose=True):
    """段階的に絞り込む。stages = [(ラウンド数, 通過数), ...]。通過数 None で全員残す。

    出題記録（群と確率ベクトル）も返す。⑦ の Bradley-Terry がこれを集計し直すので、
    追加の推論なしに別の順位が得られる。"""
    pool = list(genes)
    record, matches = {}, []
    calls = failures = exits = 0
    if verbose:
        print(f"段階 1  モデル {model}", flush=True)

    for si, (rounds, keep) in enumerate(stages, 1):
        rng = random.Random(seed + si)
        groups = balanced_groups(pool, group_size, rounds, rng)
        collected = defaultdict(list)

        for i, grp in enumerate(groups):
            try:
                res = ranker(disease, grp, model)
                calls += 1
            except Exception as e:
                failures += 1
                print(f"  S{si} round {i}: 失敗 {type(e).__name__}: {e}")
                continue
            if not res["scores"]:
                failures += 1
                continue
            if res.get("picked_exit"):
                exits += 1
            for g, pr in res["scores"].items():
                collected[g].append(pr)
            matches.append((tuple(grp), dict(res["scores"])))
            if verbose and progress_every and (i + 1) % progress_every == 0:
                print(f"    S{si}: {i + 1}/{len(groups)}", flush=True)

        means = {g: statistics.mean(v) for g, v in collected.items()}
        for g in pool:
            record[g] = {"stage": si, "mean": means.get(g),
                         "n": len(collected.get(g, []))}
        ranked = sorted(pool,
                        key=lambda g: -(means[g] if g in means else float("-inf")))
        nxt = ranked[:keep] if keep else ranked
        if verbose:
            print(f"  第 {si} 段階: {len(pool):>6} → {len(nxt):>6}  "
                  f"（{len(groups)} ラウンド、採点済み {len(means)}）", flush=True)
        pool = nxt

    return {"record": record, "final_order": pool, "matches": matches,
            "calls": calls, "failures": failures, "exits": exits}


In [ ]:
# 段階 1 の設計。(次段階へ通す割合, 1 遺伝子あたりの登場回数)。
# 最終順位は段階 2 の総当たりが付けるので、ここは 100 個まで落とすのが仕事。
# 1 段階で切る割合は 70% 程度までに抑える。登場 5 回の平均で 90% を落とすと、
# 運悪く強い群に当たり続けた本物を落とす。
STAGE_PLAN = [
    (0.30, 5),      # 1,000 → 300。各 5 回登場
    (0.33, 10),     # 300 → 100。各 10 回登場
]
RR_KEEP = 100       # 段階 2（総当たり）に送る数。N²/2 で効くので増やすと急に重い


def make_stages(n_genes, plan, group_size):
    """(通す割合, 登場回数) -> winnow() が取る (ラウンド数, 通過数) の列。

    ラウンド数 = 対象数 x 登場回数 / 群サイズ。群を小さくするとラウンド数は
    反比例して増えるが、プロンプトが短くなって 1 回が速くなるのでほぼ相殺される。
    """
    stages, pool = [], n_genes
    for frac, appear in plan:
        rounds = max(1, round(pool * appear / group_size))
        keep = None if frac is None else max(1, round(pool * frac))
        stages.append((rounds, keep))
        pool = keep if keep else pool
    return stages


STAGES = make_stages(len(SURVIVORS), STAGE_PLAN, N_PER_ROUND)
WINNOW_SEED = 0

# 1 呼び出しの実時間はその場で測る。決め打ちの定数は当てにならない。
# 中央値を取るのは、他のモデルとメモリを取り合っている間に測ると 1 回だけ
# 桁違いに遅い値が混じるため（実測で 1.0 秒の条件に 28 秒が混ざったことがある）。
# 平均だとその 1 回に引きずられて、見積もりが 40 倍ずれる。
# 段階ごとにモデルが違えば速度も違うので、モデルを引数に取る。
def measure_sec_per_call(model, pool, group_size=None, n=6, warmup=2, seed=12345):
    """1 呼び出しの実測（中央値）。測れなければ None。"""
    if not model or not pool:
        return None
    group_size = group_size or N_PER_ROUND
    rng = random.Random(seed)
    ts = []
    for k in range(n):
        grp = rng.sample(list(pool), min(group_size, len(pool)))
        t0 = time.time()
        ranker(DISEASE, grp, model)
        if k >= warmup:                     # 最初の数回はウォームアップ
            ts.append(time.time() - t0)
    sec = statistics.median(ts)
    print(f"1 呼び出しの実測（{model}）: 中央値 {sec:.2f}s  "
          f"（{min(ts):.2f}〜{max(ts):.2f}s）")
    if max(ts) > 4 * min(ts):
        print("  ⚠ ばらつきが大きすぎます。他のモデルが常駐していませんか"
              "（`ollama ps` で確認、unload() で降ろせます）。\n"
              "  この状態では見積もりも実行時間も当てになりません。")
    return sec


SEC_PER_CALL = measure_sec_per_call(RANK_MODEL, SURVIVORS) or 2.0
print()

if SURVIVORS:
    winnow_preview(len(SURVIVORS), STAGES, N_PER_ROUND, SEC_PER_CALL)


In [ ]:
win = None
if RANK_MODEL and SURVIVORS:
    win = winnow(DISEASE, SURVIVORS, RANK_MODEL, STAGES,
                 group_size=N_PER_ROUND, seed=WINNOW_SEED)
    print(f"\n呼び出し {win['calls']}  失敗 {win['failures']}")
    if USE_EXIT_OPTION:
        print(f"「その他」が 1 位だった回: {win['exits']}/{win['calls']}"
              f"（関係する遺伝子を含まない群が多いほど高くなるのが正常）")
    for g in KNOWN_ANSWERS:
        if g in SURVIVORS:
            r = win["record"].get(g, {})
            pos = win["final_order"].index(g) + 1 if g in win["final_order"] else None
            print(f"  {g:<10} 到達段階 {r.get('stage')}"
                  + (f"、段階 1 通過（{pos} 位 / {len(win['final_order'])}）"
                     if pos else "、段階 1 で脱落"))
else:
    print("モデル未選択のため実行しません。")

# ===== 段階 1 の保存 =====
# record は 1 遺伝子 1 行、matches は 1 出題 1 行。matches があれば段階 2 の結果と
# 連結して Bradley-Terry をやり直せる（推論をせずに順位の付け方だけ変えられる）。
if win:
    _final = set(win["final_order"])
    save_rows("stage1_record.tsv",
              ["gene", "protein_name", "stage_reached", "mean_score", "n_seen", "passed"],
              [[g, PROTEIN_NAMES.get(g, ""), r.get("stage"),
                None if r.get("mean") is None else round(r["mean"], 6),
                r.get("n", 0), int(g in _final)]
               for g, r in win["record"].items()])
    save_json("stage1_matches.json",
              [{"group": list(grp), "scores": {k: round(v, 6) for k, v in pr.items()}}
               for grp, pr in win["matches"]])
    record_meta(stage1={"model": RANK_MODEL, "stages": STAGES, "group_size": N_PER_ROUND,
                        "seed": WINNOW_SEED, "calls": win["calls"],
                        "failures": win["failures"], "exits": win["exits"],
                        "kept": len(win["final_order"]),
                        "use_protein_names": USE_PROTEIN_NAMES,
                        "use_gene_names": USE_GENE_NAMES,
                        "use_exit_option": USE_EXIT_OPTION})


## ⑦ Bradley-Terry レーティング

段階 1 と段階 2 の出題を**まとめて 1 つのモデルに食わせます**。段階 2 は 1 群 2 個の
出題にすぎないので、同じ枠組みでそのまま扱えます。段階 1 の対戦が上位 100 個と
それ以外をつなぐので、総当たりに出ていない遺伝子にも比較可能な数字が付きます。

平均スコアは**誰と戦って勝ったかを見ていません**。弱い 25 個に勝った 1.0 と、
DRD2 級に勝った 1.0 が同じ重みになります。5,000 個・26 個ずつなら 1 周で 192 個の勝者が出るので、
数周まわせば全勝する遺伝子が大量に生まれ、その中の順序は任意です。

Bradley-Terry（Elo と同じ族）は 1 回の出題を **強さ π の比による選択**とみなして最尤推定します。

```
P(群 G で i が選ばれる) = π_i / Σ_{j∈G} π_j
```

強い相手に勝てば π は大きく上がり、弱い相手にだけ勝っても上がりません。
**追加の推論コストはゼロ**で、⑥ の出題記録を集計し直すだけです。

真の強さが分かっている合成データ（500 個・2,000 ラウンド）での検証:

| | 真の強さとの Spearman | 同点 | 真の上位 10 の推定順位 |
|---|---|---|---|
| BT | **+0.9999** | 0 個 | 1,2,3,4,5,6,7,8,9,10 |
| 平均スコア | +0.9993 | 2 個 | 1〜9 と **13** |

実データでは、19,297 遺伝子の実行で DRD3 が 30 位→13 位、HTR6 が 79 位→18 位、
HTR3A が 100 位→19 位に上がりました。いずれも抗精神病薬の実標的で、
強い相手に当たって平均が下がっていたものです。

**一度も勝てなかった遺伝子は同点のまま並びます。** これは正しい挙動です。
全敗どうしを区別する情報はどこにも無いので、順序を付ければ捏造になります。
同点の数がそのまま、証拠が足りていない遺伝子の数です。


In [ ]:
def bradley_terry(matches, alpha=0.5, iters=300, tol=1e-9):
    """Luce/BT を MM 法で最尤推定する。

    matches は (群の遺伝子, {遺伝子: 確率}) の列。観測は「1 個が選ばれた」ではなく
    確率ベクトルなので、勝ち数は小数で数える（情報を捨てないため）。

    alpha は強さ 1 の仮想対戦相手との引き分けで、一度も勝てなかった遺伝子の
    レーティングが -inf に飛ぶのを防ぐ。"""
    W, sets = defaultdict(float), defaultdict(list)
    for grp, probs in matches:
        for g in grp:
            W[g] += probs.get(g, 0.0)
            sets[g].append(grp)
    if not W:
        return {}

    pi = {g: 1.0 for g in W}
    for _ in range(iters):
        cache, new = {}, {}
        for g in pi:
            d = 0.0
            for grp in sets[g]:
                s = cache.get(id(grp))
                if s is None:
                    s = sum(pi[x] for x in grp if x in pi)
                    cache[id(grp)] = s
                if s > 0:
                    d += 1.0 / s
            d += 2 * alpha / (pi[g] + 1.0)
            new[g] = (W[g] + alpha) / d if d > 0 else pi[g]
        gm = math.exp(statistics.mean(math.log(v) for v in new.values() if v > 0))
        new = {g: v / gm for g, v in new.items()}
        delta = max(abs(new[g] - pi[g]) for g in pi)
        pi = new
        if delta < tol:
            break
    return {g: math.log(v) for g, v in pi.items()}


## ⑧ 段階 2 — 総当たり

段階 1 を通った 100 個を、**1 対 1 で全ペア**戦わせます。ここは「その他」を入れません。
2 個から選ぶ場面で逃げ道を残すと、どちらが強いかという情報そのものが取れないためです。

**同じペアを両方向で 2 回**出します。選択肢の並び順そのものに偏りがあり、
1 方向だけだと先に置かれた側が得をします。両方向出して足せばその分が相殺されます。

比較数は N²/2 で増えます。50 個 1,225 ペア、100 個 4,950 ペア、200 個 19,900 ペア。
両方向なのでその 2 倍が呼び出し数です。**200 個を超えると総当たりは現実的でなくなる**ので、
`RR_KEEP` を増やすときは下の見積もりを見てから決めてください。

50 遺伝子（全て OT 関連上位という最難条件）での実測:

| 表示 | 時間 | 正解の順位 | 上位 5 |
|---|---|---|---|
| 記号のみ | 12.5 分 | [1, 3] | SLC7A9, SLC22A12, SLC3A1, SLC12A1, SLC14A2 |
| **記号＋蛋白質名** | 23.2 分 | **[1, 2]** | SLC3A1, SLC7A9, SLC22A12, SLC12A1, SLC12A3 |


In [ ]:
def round_robin(disease, genes, model, both_ways=True, progress_every=500):
    """全ペアを 1 対 1 で戦わせ、(群, {遺伝子: 確率}) の列を返す。

    形式は winnow() の matches と同じなので、そのまま連結して
    bradley_terry() に渡せる。

    exit_option=False、shuffle_labels=False で呼ぶ。並びは both_ways が
    受け持つので、ここで混ぜると順序の効果を打ち消せなくなる。"""
    genes = list(genes)
    pairs = list(itertools.combinations(genes, 2))
    print(f"段階 2  モデル {model}", flush=True)
    matches, failures = [], 0
    t0 = time.time()
    n = 0
    for a, b in pairs:
        for grp in (([a, b], [b, a]) if both_ways else ([a, b],)):
            n += 1
            try:
                res = ranker(disease, grp, model, shuffle_labels=False,
                             exit_option=False)
            except Exception as e:
                failures += 1
                if failures <= 3:
                    print(f"  {grp}: 失敗 {type(e).__name__}: {e}")
                continue
            if res["scores"]:
                matches.append((tuple(grp), dict(res["scores"])))
            if progress_every and n % progress_every == 0:
                el = time.time() - t0
                total = len(pairs) * (2 if both_ways else 1)
                print(f"  {n}/{total}  {el / 60:.1f}分経過 "
                      f"（残り {(total - n) * el / n / 60:.0f}分）", flush=True)
    el = time.time() - t0
    print(f"\n  {len(pairs)} ペア x {2 if both_ways else 1} 方向 = {n} 呼び出し  "
          f"{el / 60:.1f}分（1 回 {el / max(n, 1):.2f}秒、失敗 {failures}）")
    return matches


RR_BOTH_WAYS = True

FINALISTS = (win["final_order"][:RR_KEEP] if win else [])
if FINALISTS:
    _pairs = len(FINALISTS) * (len(FINALISTS) - 1) // 2
    _calls = _pairs * (2 if RR_BOTH_WAYS else 1)
    # 段階 1 と別のモデルなら速度も別。段階 1 の秒数を流用すると見積もりが外れる
    # （実測で qwen3:14b 0.60 秒に対し gemma3:27b は 5.96 秒、10 倍違った）。
    SEC_PER_CALL_RR = (SEC_PER_CALL if RR_MODEL == RANK_MODEL else
                       (measure_sec_per_call(RR_MODEL, FINALISTS, group_size=2) or SEC_PER_CALL))
    print(f"総当たり: {len(FINALISTS)} 個 = {_pairs} ペア = {_calls} 呼び出し  "
          f"見積もり {_calls * SEC_PER_CALL_RR / 60:.0f}分")


In [ ]:
# 段階 1 と違うモデルを使うなら、先に段階 1 のモデルを降ろす。
# 2 つ載ったまま走らせると載せ替えが起きて 30 倍以上遅くなる。
if RR_MODEL != RANK_MODEL:
    unload(RANK_MODEL)

rr_matches = []
if RR_MODEL and FINALISTS:
    rr_matches = round_robin(DISEASE, FINALISTS, RR_MODEL, both_ways=RR_BOTH_WAYS)

bt = {}
all_matches = (win["matches"] if win else []) + rr_matches
if all_matches:
    bt = bradley_terry(all_matches)
    print(f"\n段階 1 の {len(win['matches']) if win else 0} 出題と"
          f"段階 2 の {len(rr_matches)} 出題、合わせて {len(all_matches)} 件から"
          f" {len(bt)} 遺伝子のレーティングを推定（追加の推論なし）")
    ties = Counter(round(v, 6) for v in bt.values())
    print(f"同点: {sum(c for c in ties.values() if c > 1)} 個"
          f"（証拠が足りていない遺伝子の数）")

# ===== 段階 2 の保存 =====
if rr_matches:
    save_json("stage2_matches.json",
              [{"group": list(grp), "scores": {k: round(v, 6) for k, v in pr.items()}}
               for grp, pr in rr_matches])
    record_meta(stage2={"model": RR_MODEL, "n_genes": len(FINALISTS),
                        "both_ways": RR_BOTH_WAYS, "calls": len(rr_matches),
                        "rr_keep": RR_KEEP})
if bt:
    save_rows("bradley_terry.tsv", ["gene", "protein_name", "log_strength", "elo"],
              [[g, PROTEIN_NAMES.get(g, ""), round(v, 6),
                round(1500 + 400 * v / math.log(10), 1)]
               for g, v in sorted(bt.items(), key=lambda kv: -kv[1])])


## ⑨ 結果

| 列 | 意味 |
|---|---|
| `rank` | 到達段階が第一キー、その中が Bradley-Terry レーティング順 |
| `elo` | 400 点差で 10 倍強いという目盛り |
| `screen` | 段階 0 の `logP(Yes) − logP(No)`。大きいほど「機序がある」側 |
| `final` | 段階 2（総当たり）まで残ったか |
| `stage` | 段階 1 の何段目まで進んだか |
| `mean_score` | 段階 1 での平均スコア。**段階をまたいで比べてはいけません** |
| `n_seen` | 段階 1 で出題された回数 |

並べ替えを BT だけにしないのは、**段階 2 で何度も戦って負けた遺伝子が、
段階 1 で数回しか出ていない遺伝子より下に来てしまう**ためです。後者は証拠が無いので
アンカー（1500 点）付近に留まるだけで沈みません。同じ `rank` が「絞り込みの結果」と
「対戦の結果」の 2 つを指さないよう、層を分けています。

**表に出るのは段階 0 を通った候補だけです。** 落ちた遺伝子は一度も比較されていないので、
順位を付けると測っていないものに順位が付いてしまいます。
`screen` の値だけはその全員について残っているので、気になる遺伝子は
`SCREEN_MARGIN` を直接引いてください。


In [ ]:
def result_rows(win, bt, genes, margins=None, finalists=()):
    if not win:
        return []
    margins = margins or {}
    final = set(finalists)
    order = {g: i for i, g in enumerate(win["final_order"])}
    rows = []
    for g in genes:
        r = win["record"].get(g, {})
        rows.append({"gene": g, "stage": r.get("stage"),
                     "mean_score": None if r.get("mean") is None else round(r["mean"], 5),
                     "n_seen": r.get("n", 0),
                     "screen": None if g not in margins else round(margins[g], 2),
                     "final": g in final,
                     "_o": order.get(g, 10 ** 9)})
    rows.sort(key=lambda r: (-(r["stage"] or 0), r["_o"],
                             -(r["mean_score"] if r["mean_score"] is not None else -1)))
    for i, r in enumerate(rows, 1):
        r["mean_rank"] = i
        r.pop("_o")
    # 未採点（n_seen=0）は最後尾に固定する。スコアが無いものを
    # 順位の途中に混ぜない。
    rows.sort(key=lambda r: (0 if r.get("n_seen") else 1,))
    if bt:
        elo = {g: 1500 + 400 * (v / math.log(10)) for g, v in bt.items()}
        for r in rows:
            r["elo"] = round(elo.get(r["gene"], float("nan")), 1)
        # 並べ替えは「どこまで進んだか」が第一キー、BT はその中の順序。
        # BT だけで並べると、段階 2 で 100 回戦って負けた遺伝子が、段階 1 で
        # 4 回しか出ていない遺伝子より下に来る。前者のほうが証拠は多いのに、
        # 後者はアンカー（1500 点）付近に留まるだけで沈まないため。
        # 同じ表の中で rank が 2 つの意味を持たないよう、層を分ける。
        rows.sort(key=lambda r: (0 if r["final"] else 1,
                                 -(r["stage"] or 0),
                                 -(bt.get(r["gene"], float("-inf")))))
        for i, r in enumerate(rows, 1):
            r["rank"] = i
            r["move"] = r["mean_rank"] - i
    else:
        for r in rows:
            r["rank"] = r["mean_rank"]
    return rows


rows = result_rows(win, bt, SURVIVORS, SCREEN_MARGIN, FINALISTS)
cols = ["rank", "gene", "elo", "screen", "final", "stage", "mean_score",
        "n_seen", "mean_rank", "move"]
cols = [c for c in cols if rows and c in rows[0]]

if rows and HAVE_PANDAS:
    df = pd.DataFrame(rows)[cols]
    try:
        display(df.head(30))
    except NameError:
        print(df.head(30).to_string(index=False))
elif rows:
    print("  ".join(cols))
    for r in rows[:30]:
        print("  ".join(str(r.get(c, "")) for c in cols))
else:
    print("結果がありません。")

dropped = len(GENES) - len(SURVIVORS)
if dropped:
    print(f"\n段階 0 で {dropped}/{len(GENES)} 遺伝子を落としました（表に出ません）。")

# 一度も出題されなかった遺伝子は順位を持たない。順位を付けて出すと、
# 「4362 位」が実力のように見えてしまう（実際は測っていないだけ）。
n_unscored = sum(1 for r in rows if not r.get("n_seen"))
if n_unscored:
    print(f"\n⚠ {n_unscored}/{len(rows)} 遺伝子は段階 1 で一度も出題されていません（順位なし）。"
          f"\n  STAGE_PLAN の登場回数を増やすか、SCREEN_KEEP を減らしてください。")

if rows and KNOWN_ANSWERS:
    info = {r["gene"]: r for r in rows}
    ranked = [r for r in rows if r.get("n_seen")]
    print("\n答え合わせ:")
    for g in KNOWN_ANSWERS:
        r = info.get(g)
        if g not in GENES:
            print(f"  {g:<10}（候補に無い）")
        elif r is None:
            print(f"  {g:<10}⚠ 段階 0 で脱落"
                  + (f"（logit差 {SCREEN_MARGIN[g]:+.2f}）" if g in SCREEN_MARGIN else ""))
        elif not r.get("n_seen"):
            print(f"  {g:<10}順位なし — 段階 1 で一度も出題されていません")
        else:
            print(f"  {g:<10}{r['rank']} 位 / {len(rows)}"
                  f"  段階 2 {'到達' if r.get('final') else '未到達'}"
                  + (f"  screen {r['screen']:+.2f}" if r.get("screen") is not None else ""))


### 保存


In [ ]:
# 最終結果。段階 0 で落ちた遺伝子は入りません（stage0_screen.tsv に全件あります）。
if rows:
    name = f"{DISEASE.replace(' ', '_')}_ranking.csv"
    out = os.path.join(RUN_DIR, name) if RUN_DIR else name
    with open(out, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=cols, extrasaction="ignore")
        w.writeheader()
        w.writerows(rows)
    print("書き出しました:", out)
    record_meta(finished=time.strftime("%Y-%m-%d %H:%M:%S"), result_csv=name)
    if RUN_DIR:
        print("\nこの実行の成果物:")
        for f in sorted(os.listdir(RUN_DIR)):
            print(f"  {os.path.join(RUN_DIR, f)}"
                  f"  {os.path.getsize(os.path.join(RUN_DIR, f)) / 1024:.0f}KB")


## 数字を信じる前に

- **段階 0 で落ちた遺伝子は「関係ない」と判定されたのではありません。**
  上位 `SCREEN_KEEP` 個に入らなかっただけです。閾値ではなく順位で切っているので、
  候補リストが大きいほど切られる数も増えます。
- **同族で機能の近い遺伝子は最後まで混ざります。** cystinuria の総当たりでも 3 位は
  SLC22A12（尿酸トランスポーター、無関係）です。上位数個を仮説の種として読むもので、
  順位表をそのまま信じるものではありません。
- **`n_seen` が小さい行は読まないでください。** 登場数回の平均は、どの 25 個と同じ群に
  入ったかでいくらでも動きます。
- **スコアは相対値です。** 弱い候補ばかりの群に入れば弱い遺伝子でも 1 位になります。
  絶対的な確信度ではありません。
- **対照を取らないと意味は分かりません。** 疾患名を無関係なものに差し替えて同じ順位が出るなら、
  疾患ではなく知名度を読んでいます。`DISEASE` を変えて 1 度まわしてみてください。
- **候補リストの作り方が結果を決めます。** 無作為抽出だけで 10,000 個を選んだときは、
  HTR2A・DRD4・CHRM1 といった主要標的が候補から漏れていました。
  同梱のリストは Open Targets の関連遺伝子と承認薬の標的を必ず含む構成にしてあります。
- **「どれでもない」の選択肢は入れていません。** 偽陽性は完全に消えますが
  （無関係な群で 12/12 棄却）、既知の正解も一緒に消えます（DRD2 級を拾えたのは 1/10、
  棄却例を few-shot に足しても 3/10）。偽陽性を偽陰性に置き換えるだけでした。
- **対照疾患との差分も入れていません。** `p(遺伝子|対象) − p(遺伝子|human disease)` を引くと、
  本物の疾患遺伝子ほど罰せられます。「human disease に関連する遺伝子」は求めている信号そのもので、
  それを引けば信号が消えます。これで DRD2 が 106 位まで落ちました。
